# Flyguys: fly brain engine on a GPU

This notebook runs the project's batched fly-brain simulator (`brain/`) on Colab's free T4 GPU. It follows on from `fly_brain_starter.ipynb`: same connectome, same neuron groups, same settings from Shiu et al. 2024, but written in PyTorch so that hundreds of brains run side by side.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom.

No FlyWire token is needed.

## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Use Runtime -> Change runtime type -> T4 GPU, then run again."
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 2. Get the code and the connectome
Clones the project and downloads Shiu's connectivity file (about 100 MB).

In [ ]:
import os
REPO = "https://github.com/Vsbs14/Flyguys.git"
BRANCH = "phase-1-2"  # set back to "main" once this branch is merged
if not os.path.exists("/content/Flyguys"):
    !git clone -q --branch {BRANCH} {REPO} /content/Flyguys
%cd /content/Flyguys
!git pull -q
!git log --oneline -1
!python scripts/fetch_data.py

## 3. Is the simulator faithful to Shiu's model?
A small random network was run in Shiu's original Brian2 model and its spikes saved. This replays it in our PyTorch simulator and compares every spike. It runs on the CPU and takes about a minute.

In [ ]:
!python -m brain.test_equivalence

## 4. The six sanity checks, on the GPU
| # | Check |
|---|---|
| 1 | No input → silence |
| 2 | Sugar → MN9 fires |
| 3 | Sugar + bitter → MN9 drops a lot |
| 4 | Looming → Giant Fibre fires |
| 5 | Giant Fibre silenced → no escape |
| 6 | Input stopped → activity dies out |

In [ ]:
!python -m brain.sanity --device cuda --replicates 8

## 5. Benchmark
Times one batch of 200 ms decisions at 64, 256 and 1,024 brains. This sets the compute budget for the response tables and the lesion screen.

In [ ]:
!python -m brain.bench --device cuda

## 5b. Decision rule (Phase 2)
Turns five sense intensities (sugar, bitter, looming, obstacle, target; 0 to 1 each) into one action: idle, feed, escape, retreat or advance. This calibrates each output's reference rate, then checks the rule on fixed scenarios such as *sugar + looming -> escape*. It takes about a minute.

In [ ]:
!python -m brain.test_interface
!python -m brain.calibrate --device cuda

## 6. Results
Shows the saved tables and downloads them as a zip. Unzip it into the repo so `results/` holds the GPU runs. The calibrated rule itself is saved to `data/decision_rule.json`; copy it too.

In [ ]:
from IPython.display import Markdown, display
for name in ["sanity_intact_cuda", "benchmark_cuda", "decision_calibration"]:
    display(Markdown(open(f"results/{name}.md").read()))

In [ ]:
from google.colab import files
!zip -qr /content/flyguys_results.zip results data/decision_rule.json
files.download("/content/flyguys_results.zip")

## 7. Use the engine yourself
Every column of the state is one independent brain. Here 5 brains get 5 different sugar levels in a single run.

In [ ]:
import torch
from brain import Connectome, NeuronGroups, BrainSim

conn = Connectome.load()
groups = NeuronGroups(conn)
sim = BrainSim(conn, device="cuda")

sugar_hz = torch.tensor([0., 50., 100., 150., 200.])
state = sim.new_state(n_brains=len(sugar_hz), seed=0)
res = sim.run(state, duration_ms=200,
              inputs=[(groups.sense("sugar"), sugar_hz)],
              monitor=groups.output("feed"))
for hz, mn9 in zip(sugar_hz, res.rates_hz.mean(0)):
    print(f"sugar {hz:5.0f} Hz  ->  MN9 {mn9:5.1f} Hz")

## 8. Ask the brain for decisions
Each row is one colony's senses (sugar, bitter, looming, obstacle, target).

In [ ]:
from brain.interface import FlyBrain, DecisionRule

brain = FlyBrain(sim, groups, DecisionRule.load())
worlds = {
    "food":            [1, 0, 0, 0, 0],
    "toxic food":      [1, 1, 0, 0, 0],
    "food + threat":   [1, 0, 1, 0, 0],
    "wall ahead":      [0, 0, 0, 1, 0],
    "something to chase": [0, 0, 0, 0, 1],
}
d = brain.decide(list(worlds.values()), seed=0)
for name, action in zip(worlds, d.actions):
    print(f"{name:20s} -> {action}")